# 🚀 Improved Chunking Pipeline (với Dataset-Specific Optimization)

## 📋 Overview

Pipeline này sử dụng **dataset-specific chunking strategies** đã được tối ưu từ **Notebook 3**, kết hợp với **dataset-specific retrieval parameters**:

### 🎯 Key Features:

1. **Semantic Chunking** ✨ (NEW!)
   - Sử dụng **semantic chunking** cho hầu hết datasets
   - Dựa trên evaluation trong notebook 3
   - Tự động load pre-chunked corpus

2. **Hybrid Retrieval** 🔍
   - Dense retrieval (BAAI/bge-large-en-v1.5)
   - BM25 (lexical matching) - **tốt cho numerical queries**
   - **Dataset-specific alpha** weighting

3. **Advanced Reranking** 📊
   - BAAI/bge-reranker-v2-m3
   - Re-score top candidates

4. **Smart Aggregation** 🔗
   - Chunk-level retrieval
   - Document-level aggregation (max score)

5. **🆕 Dataset-Specific Optimizations** 🎯
   - **MultiHeirTT**: top_k=200, 60% BM25 (numerical reasoning)
   - **TATQA**: top_k=150, 50-50 hybrid
   - **FinQA/ConvFinQA**: top_k=120, slight BM25 boost

### 📂 Pre-Chunked Data Source (từ Notebook 3):

```
../../data/chunked_corpus/
├── convfinqa_corpus_chunked_optimal.jsonl      (semantic 2000/0.6)   ✨
├── financebench_corpus_chunked_optimal.jsonl   (semantic 1000/0.7)   ✨
├── finder_corpus_chunked_optimal.jsonl         (recursive 512/50)
├── finqa_corpus_chunked_optimal.jsonl          (semantic 2500/0.65)  ✨
├── finqabench_corpus_chunked_optimal.jsonl     (preserve_tables 1024/100)
├── multiheirtt_corpus_chunked_optimal.jsonl    (semantic 3000/0.65)  ✨
├── tatqa_corpus_chunked_optimal.jsonl          (semantic 2000/0.7)   ✨
└── best_chunking_config_per_dataset.json       (configs)
```

### ⚡ Performance Improvements:

| Dataset | Baseline | With Optimization | Improvement |
|---------|----------|-------------------|-------------|
| TATQA | 0.3718 | 0.5335 | **+43.5%** |
| MultiHeirTT | 0.1630 | 0.2131 | **+30.7%** |
| FinanceBench | - | 0.7938 | semantic ✨ |
| FinQABench | - | 0.8676 | preserve_tables |

---

## 📦 Installation

In [ ]:
# Install required packages
!pip install -q sentence-transformers faiss-cpu FlagEmbedding rank-bm25

## 📚 Imports

In [1]:
# Core Libraries
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from typing import List, Dict, Tuple
import re

# Embedding & Retrieval
from sentence_transformers import SentenceTransformer
import faiss

# Reranking
from FlagEmbedding import FlagReranker

# BM25
from rank_bm25 import BM25Okapi

# Utils
import warnings
warnings.filterwarnings('ignore')
import logging
logging.disable(logging.CRITICAL)

In [2]:
# Check GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("Running on CPU")
    device = 'cpu'

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## ⚙️ Configuration (với Dataset-Specific Overrides)

In [ ]:
# Load configuration from config.py
from config import CONFIG, DATASET_SPECIFIC_CONFIG, print_config

# Print configuration summary
print_config()

✅ Configuration loaded

🎯 CHUNKING STRATEGY:
   ✅ Using PRE-CHUNKED corpus from notebook 2
   📂 Source: ../data/chunked_corpus
   📋 Config: ../data/chunked_corpus/best_chunking_config_per_dataset.json
   ℹ️ Each dataset uses its OPTIMAL chunking method

🔗 Aggregation: max (chunks → docs)

🔧 DATASET-SPECIFIC OVERRIDES:
   multiheirtt: {'top_k_retrieval': 200, 'top_k_rerank': 80, 'hybrid_alpha': 0.4}
   tatqa: {'top_k_retrieval': 150, 'top_k_rerank': 60, 'hybrid_alpha': 0.5}
   finqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   convfinqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}


## 🔧 Helper Functions

In [ ]:
# Import shared utilities
import sys
sys.path.insert(0, '..')
from utils import (
    load_jsonl_data,
    load_prechunked_corpus,
    QRELS_MAPPING,
    DATASETS
)

print("✅ Data loading functions imported from utils.py")

✅ Data loading functions defined


## ✅ Verify Pre-Chunked Data (Semantic Chunking từ Notebook 2)

Kiểm tra pre-chunked data đã được tạo từ notebook 2 với **semantic chunking**:

In [5]:
import json
import os

# Check pre-chunked files
chunked_dir = CONFIG['chunked_corpus_dir']
config_file = CONFIG['chunking_config_file']

print("="*80)
print("🔍 VERIFICATION: Pre-Chunked Data from Notebook 2")
print("="*80)

# Check if config file exists
if os.path.exists(config_file):
    print(f"\n✅ Config file found: {config_file}")
    
    with open(config_file, 'r') as f:
        chunk_configs = json.load(f)
    
    print(f"\n📋 Dataset-Specific Chunking Configurations:")
    print("-"*80)
    print(f"{'Dataset':<15} {'Method':<25} {'NDCG@10':<12} {'Status':<10}")
    print("-"*80)
    
    for dataset in CONFIG['datasets']:
        if dataset in chunk_configs:
            cfg = chunk_configs[dataset]
            method = cfg['method']
            
            # Format method string
            if method != 'no_chunking' and cfg.get('chunk_size'):
                method_str = f"{method} ({cfg['chunk_size']}/{cfg['chunk_overlap']})"
            else:
                method_str = method
            
            ndcg = cfg.get('ndcg_10', 0.0)
            
            # Check if chunked file exists
            chunked_file = os.path.join(chunked_dir, f"{dataset}_corpus_chunked_optimal.jsonl")
            status = "✅ Ready" if os.path.exists(chunked_file) else "❌ Missing"
            
            print(f"{dataset:<15} {method_str:<25} {ndcg:<12.4f} {status:<10}")
        else:
            print(f"{dataset:<15} {'N/A':<25} {'N/A':<12} {'❌ Missing':<10}")
    
    print("-"*80)
    
    # Count available files
    available = sum(1 for d in CONFIG['datasets'] 
                   if os.path.exists(os.path.join(chunked_dir, f"{d}_corpus_chunked_optimal.jsonl")))
    print(f"\n📊 Summary: {available}/{len(CONFIG['datasets'])} datasets have pre-chunked corpus")
    
    if available == len(CONFIG['datasets']):
        print("✅ All datasets ready! Pipeline will use pre-chunked data.")
    else:
        print("⚠️ Some datasets missing. Pipeline will chunk on-the-fly for missing datasets.")
else:
    print(f"\n❌ Config file not found: {config_file}")
    print("⚠️ Pipeline will use fallback chunking settings.")

print("\n" + "="*80)

🔍 VERIFICATION: Pre-Chunked Data from Notebook 2

✅ Config file found: ../data/chunked_corpus/best_chunking_config_per_dataset.json

📋 Dataset-Specific Chunking Configurations:
--------------------------------------------------------------------------------
Dataset         Method                    NDCG@10      Status    
--------------------------------------------------------------------------------
convfinqa       semantic (2000/0.6)       0.6591       ✅ Ready   
financebench    semantic (1000/0.7)       0.7938       ✅ Ready   
finder          recursive (512/50)        0.7046       ✅ Ready   
finqa           semantic (2500/0.65)      0.6284       ✅ Ready   
finqabench      preserve_tables (1024/100) 0.8676       ✅ Ready   
multiheirtt     semantic (3000/0.65)      0.2131       ✅ Ready   
tatqa           semantic (2000/0.7)       0.5335       ✅ Ready   
--------------------------------------------------------------------------------

📊 Summary: 7/7 datasets have pre-chunked corpus
✅ 

## ✂️ Optimal Chunking Functions

In [6]:
def detect_tables(text: str) -> List[Tuple[int, int]]:
    """Detect table regions using heuristics"""
    lines = text.split('\n')
    table_regions = []
    in_table = False
    table_start = 0
    
    for i, line in enumerate(lines):
        # Detect table markers: pipes, tabs, or multiple spaces
        is_table = (
            line.count('|') >= 2 or
            line.count('\t') >= 2 or
            len(re.findall(r'\s{3,}', line)) >= 2
        )
        
        if is_table and not in_table:
            in_table = True
            table_start = max(0, i - 1)  # Include line before
        elif not is_table and in_table:
            in_table = False
            table_end = min(len(lines), i + 1)  # Include line after
            if table_end - table_start >= 3:  # At least 3 lines
                table_regions.append((table_start, table_end))
    
    if in_table:
        table_regions.append((table_start, len(lines)))
    
    return table_regions


def chunk_text_fixed(text: str, chunk_size: int, overlap: int) -> List[str]:
    """Fixed-size character chunking with overlap"""
    if len(text) <= chunk_size:
        return [text]
    
    chunks = []
    step = chunk_size - overlap
    
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if len(chunk.strip()) >= 100:  # Minimum chunk size
            chunks.append(chunk.strip())
        
        # Stop if we've reached the end
        if i + chunk_size >= len(text):
            break
    
    return chunks


def chunk_document_optimal(doc_id: str, title: str, text: str, 
                          chunk_size: int, overlap: int, preserve_tables: bool):
    """Optimal chunking: fixed-size with table preservation"""
    chunks = []
    full_text = f"{title}\n{text}" if title else text
    
    # For short documents, no chunking needed
    if len(full_text) < chunk_size * 1.5:
        chunks.append({
            'chunk_id': f"{doc_id}_c0",
            'text': full_text,
            'doc_id': doc_id,
            'is_table': False
        })
        return chunks
    
    # Detect tables
    if preserve_tables:
        lines = full_text.split('\n')
        table_regions = detect_tables(full_text)
        
        if table_regions:
            chunk_idx = 0
            prev_end = 0
            
            for table_start, table_end in table_regions:
                # Chunk text before table
                if table_start > prev_end:
                    before_text = '\n'.join(lines[prev_end:table_start])
                    if before_text.strip():
                        for chunk_text in chunk_text_fixed(before_text, chunk_size, overlap):
                            chunks.append({
                                'chunk_id': f"{doc_id}_c{chunk_idx}",
                                'text': chunk_text,
                                'doc_id': doc_id,
                                'is_table': False
                            })
                            chunk_idx += 1
                
                # Keep table as single chunk
                table_text = '\n'.join(lines[table_start:table_end])
                chunks.append({
                    'chunk_id': f"{doc_id}_t{chunk_idx}",
                    'text': f"[TABLE]\n{table_text}",
                    'doc_id': doc_id,
                    'is_table': True
                })
                chunk_idx += 1
                prev_end = table_end
            
            # Chunk text after last table
            if prev_end < len(lines):
                after_text = '\n'.join(lines[prev_end:])
                if after_text.strip():
                    for chunk_text in chunk_text_fixed(after_text, chunk_size, overlap):
                        chunks.append({
                            'chunk_id': f"{doc_id}_c{chunk_idx}",
                            'text': chunk_text,
                            'doc_id': doc_id,
                            'is_table': False
                        })
                        chunk_idx += 1
            
            return chunks
    
    # No tables detected or preservation disabled - simple fixed chunking
    for i, chunk_text in enumerate(chunk_text_fixed(full_text, chunk_size, overlap)):
        chunks.append({
            'chunk_id': f"{doc_id}_c{i}",
            'text': chunk_text,
            'doc_id': doc_id,
            'is_table': False
        })
    
    return chunks


print("✅ Optimal chunking functions defined")

✅ Optimal chunking functions defined


## 🔍 Retrieval Functions

In [ ]:
# Import retrieval utilities from shared utils
from utils import normalize_scores, hybrid_search, aggregate_chunk_scores

print("✅ Retrieval functions imported from utils.py")

✅ Retrieval functions defined


## 📊 Evaluation Functions

In [ ]:
# Import evaluation functions from shared utils
from utils import compute_ndcg_batch as compute_ndcg, evaluate_results_df as evaluate_results

print("✅ Evaluation functions imported from utils.py")
print("   📋 compute_ndcg: computes average NDCG@k across multiple queries")
print("   📋 evaluate_results: evaluates results DataFrame against qrels")

✅ Evaluation functions defined


## 🤖 Load Models

In [9]:
print("Loading models...")

# Embedding model
print(f"\n1. Loading embedding model: {CONFIG['embedding_model']}")
embed_model = SentenceTransformer(CONFIG['embedding_model'], device=device)
print("   ✅ Done")

# Reranker model
print(f"\n2. Loading reranker: {CONFIG['reranker_model']}")
reranker = FlagReranker(CONFIG['reranker_model'], use_fp16=(device=='cuda'))
print("   ✅ Done")

print("\n✅ All models loaded!")

Loading models...

1. Loading embedding model: BAAI/bge-large-en-v1.5
   ✅ Done

2. Loading reranker: BAAI/bge-reranker-v2-m3
   ✅ Done

✅ All models loaded!


## 🔄 Main Pipeline (với Dataset-Specific Optimization)

Pipeline xử lý từng dataset với:
- **Semantic chunking** từ notebook 2
- **Dataset-specific retrieval parameters** (top_k, hybrid_alpha)
- **Hybrid search** (Dense + BM25) với tuned alpha

In [10]:
def process_dataset_with_optimal_chunking(dataset_name: str, config: Dict, dataset_overrides: Dict = None):
    """Process dataset with optimal chunking strategy and dataset-specific overrides"""
    print(f"\n{'='*70}")
    print(f"📊 Processing: {dataset_name.upper()}")
    print(f"{'='*70}")
    
    # Apply dataset-specific overrides
    effective_config = config.copy()
    if dataset_overrides and dataset_name in dataset_overrides:
        overrides = dataset_overrides[dataset_name]
        effective_config.update(overrides)
        print(f"🔧 Dataset-specific overrides applied:")
        for k, v in overrides.items():
            print(f"   {k}: {v}")
    
    # Load data
    corpus_df, queries_df, qrels_df = load_jsonl_data(dataset_name, effective_config['data_dir'])
    
    # Step 1: Load or Create Chunks
    all_chunks = []
    chunk_to_doc = {}
    
    # Try to load pre-chunked data first (from notebook 2)
    chunking_method_used = None
    if effective_config.get('use_prechunked', False):
        print(f"\n📂 Loading pre-chunked corpus from notebook 2...")
        print(f"   Source: {effective_config['chunked_corpus_dir']}")
        
        all_chunks, chunking_method_used = load_prechunked_corpus(
            dataset_name, 
            effective_config['chunked_corpus_dir'],
            effective_config.get('chunking_config_file')
        )
        
        if all_chunks:
            # Build chunk-to-doc mapping (pre-chunked format uses 'original_id')
            for c in all_chunks:
                chunk_id = c.get('_id', c.get('chunk_id', ''))
                doc_id = c.get('original_id', c.get('doc_id', chunk_id))
                chunk_to_doc[chunk_id] = doc_id
            
            expansion_ratio = len(all_chunks) / len(corpus_df)
            print(f"   📈 Expansion: {len(corpus_df)} docs → {len(all_chunks)} chunks ({expansion_ratio:.2f}x)")
        else:
            print(f"   ⚠️ Pre-chunked data not found, will chunk on-the-fly")
            effective_config['use_prechunked'] = False
    
    # Fallback: chunk on-the-fly if pre-chunked not available
    if not effective_config.get('use_prechunked', False) or not all_chunks:
        print(f"\n✂️ Chunking corpus on-the-fly...")
        print(f"   Method: {effective_config['chunking_method']}")
        print(f"   Size: {effective_config['chunk_size']} chars, Overlap: {effective_config['chunk_overlap']} chars")
        
        if effective_config['use_chunking']:
            for _, row in tqdm(corpus_df.iterrows(), total=len(corpus_df), desc="Chunking"):
                chunks = chunk_document_optimal(
                    row['_id'], 
                    str(row.get('title', '')), 
                    str(row.get('text', '')),
                    effective_config['chunk_size'], 
                    effective_config['chunk_overlap'], 
                    effective_config['preserve_tables']
                )
                for c in chunks:
                    all_chunks.append(c)
                    chunk_to_doc[c['chunk_id']] = c['doc_id']
            
            num_tables = sum(1 for c in all_chunks if c.get('is_table', False))
            expansion_ratio = len(all_chunks) / len(corpus_df)
            print(f"   ✅ Created {len(all_chunks)} chunks from {len(corpus_df)} docs")
            print(f"   📈 Expansion: {expansion_ratio:.2f}x")
            print(f"   📊 Tables preserved: {num_tables}")
        else:
            for _, row in corpus_df.iterrows():
                doc_id = row['_id']
                text = f"[{row.get('title', '')}] {row.get('text', '')}"
                all_chunks.append({'chunk_id': doc_id, 'text': text, 'doc_id': doc_id})
                chunk_to_doc[doc_id] = doc_id
    
    # Extract texts and IDs (handle both formats)
    chunk_texts = [c.get('text', '') for c in all_chunks]
    chunk_ids = [c.get('_id', c.get('chunk_id', '')) for c in all_chunks]
    
    # Ensure chunk_to_doc mapping is complete
    if not chunk_to_doc:
        for c in all_chunks:
            cid = c.get('_id', c.get('chunk_id', ''))
            did = c.get('original_id', c.get('doc_id', cid))
            chunk_to_doc[cid] = did
    
    # Step 2: Embedding
    print(f"\n🔢 Encoding chunks...")
    chunk_embeddings = embed_model.encode(
        chunk_texts, 
        batch_size=effective_config['embed_batch_size'],
        show_progress_bar=True, 
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    print(f"   ✅ Encoded {len(chunk_embeddings)} chunks")
    
    # Step 3: Build FAISS Index
    print(f"\n🔍 Building FAISS index...")
    index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
    index.add(chunk_embeddings.astype('float32'))
    print(f"   ✅ Index built with {index.ntotal} vectors")
    
    # Step 4: Build BM25 Index (for hybrid)
    bm25 = None
    if effective_config['use_hybrid']:
        print(f"\n🔤 Building BM25 index...")
        tokenized = [t.lower().split() for t in chunk_texts]
        bm25 = BM25Okapi(tokenized)
        print(f"   ✅ BM25 index built")
        print(f"   ⚖️ Hybrid alpha: {effective_config['hybrid_alpha']} (Dense) / {1-effective_config['hybrid_alpha']:.1f} (BM25)")
    
    # Free memory
    del chunk_embeddings
    if device == 'cuda':
        torch.cuda.empty_cache()
    
    # Step 5: Process Queries
    print(f"\n🎯 Processing queries...")
    query_texts = [str(r.get('text', '')) for _, r in queries_df.iterrows()]
    query_ids = queries_df['_id'].tolist()
    
    query_embeddings = embed_model.encode(
        query_texts, 
        batch_size=effective_config['embed_batch_size'],
        show_progress_bar=True, 
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    print(f"   ✅ Encoded {len(query_embeddings)} queries")
    print(f"   📋 Retrieval config: top_k_retrieval={effective_config['top_k_retrieval']}, top_k_rerank={effective_config['top_k_rerank']}")
    
    # Step 6: Retrieve & Rerank
    print(f"\n🔎 Retrieving and reranking...")
    results = []
    
    for i, query_id in enumerate(tqdm(query_ids, desc="Retrieve+Rerank")):
        query_emb = query_embeddings[i]
        query_text = query_texts[i]
        
        # Hybrid or dense retrieval
        if effective_config['use_hybrid'] and bm25:
            scores, chunk_indices = hybrid_search(
                query_emb, query_text, index, bm25, chunk_texts,
                effective_config['top_k_retrieval'], effective_config['hybrid_alpha']
            )
        else:
            scores, chunk_indices = index.search(
                query_emb.reshape(1, -1).astype('float32'),
                effective_config['top_k_retrieval']
            )
            scores, chunk_indices = scores[0], chunk_indices[0]
        
        # Aggregate chunks to documents
        doc_scores = {}
        for idx, score in zip(chunk_indices, scores):
            doc_id = chunk_to_doc[chunk_ids[idx]]
            if doc_id not in doc_scores:
                doc_scores[doc_id] = []
            doc_scores[doc_id].append(float(score))
        
        doc_agg = aggregate_chunk_scores(doc_scores, effective_config['chunk_aggregation'])
        sorted_docs = sorted(doc_agg.items(), key=lambda x: x[1], reverse=True)[:effective_config['top_k_rerank']]
        
        # Rerank top documents
        candidate_ids = [d[0] for d in sorted_docs]
        candidate_texts = [
            str(corpus_df[corpus_df['_id']==d]['text'].values[0])[:2048]
            for d in candidate_ids
        ]
        
        pairs = [[query_text, t] for t in candidate_texts]
        rerank_scores = reranker.compute_score(pairs)
        
        if not isinstance(rerank_scores, list):
            rerank_scores = [rerank_scores]
        
        scored = list(zip(candidate_ids, rerank_scores))
        scored.sort(key=lambda x: x[1], reverse=True)
        
        # Store top-k final results
        for doc_id, score in scored[:effective_config['top_k_final']]:
            results.append({
                'query_id': query_id,
                'corpus_id': doc_id,
                'score': float(score)
            })
    
    results_df = pd.DataFrame(results)
    print(f"   ✅ Generated {len(results_df)} results")
    
    # Step 7: Evaluate
    eval_metrics = {}
    if effective_config['eval_on_qrels'] and qrels_df is not None:
        print(f"\n📊 Evaluating...")
        eval_metrics = evaluate_results(results_df, qrels_df)
        print(f"   NDCG@10: {eval_metrics['NDCG@10']:.4f}")
        print(f"   Queries evaluated: {eval_metrics['num_queries']}")
    
    # Clean up
    del query_embeddings, index
    if device == 'cuda':
        torch.cuda.empty_cache()
    
    return results_df, eval_metrics


print("✅ Pipeline function defined (with dataset-specific overrides support)")

✅ Pipeline function defined (with dataset-specific overrides support)


## 🚀 Run Complete Pipeline

Chạy pipeline với:
- **Semantic chunking** cho ConvFinQA, FinanceBench, FinQA, MultiHeirTT, TATQA
- **Dataset-specific parameters** cho MultiHeirTT (BM25-heavy), TATQA (balanced)

In [11]:
all_results = []
all_eval = {}
failed = []

print("\n" + "="*70)
print("🚀 STARTING OPTIMAL CHUNKING PIPELINE")
print("="*70)
print("\n📋 Using dataset-specific configurations:")
for ds in CONFIG['datasets']:
    if ds in DATASET_SPECIFIC_CONFIG:
        print(f"   🔧 {ds}: {DATASET_SPECIFIC_CONFIG[ds]}")
    else:
        print(f"   📌 {ds}: default config")

for dataset in CONFIG['datasets']:
    try:
        df_res, metrics = process_dataset_with_optimal_chunking(
            dataset, 
            CONFIG, 
            DATASET_SPECIFIC_CONFIG  # Pass dataset-specific overrides
        )
        all_results.append(df_res)
        if metrics:
            all_eval[dataset] = metrics
    except Exception as e:
        print(f"\n❌ Error processing {dataset}: {e}")
        import traceback
        traceback.print_exc()
        failed.append(dataset)

print(f"\n{'='*70}")
print(f"✅ Pipeline completed: {len(all_results)}/{len(CONFIG['datasets'])} datasets")
if failed:
    print(f"❌ Failed datasets: {failed}")
print(f"{'='*70}")


🚀 STARTING OPTIMAL CHUNKING PIPELINE

📋 Using dataset-specific configurations:
   🔧 convfinqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   📌 financebench: default config
   📌 finder: default config
   🔧 finqa: {'top_k_retrieval': 120, 'hybrid_alpha': 0.55}
   📌 finqabench: default config
   🔧 multiheirtt: {'top_k_retrieval': 200, 'top_k_rerank': 80, 'hybrid_alpha': 0.4}
   🔧 tatqa: {'top_k_retrieval': 150, 'top_k_rerank': 60, 'hybrid_alpha': 0.5}

📊 Processing: CONVFINQA
🔧 Dataset-specific overrides applied:
   top_k_retrieval: 120
   hybrid_alpha: 0.55
  Loaded 2066 docs, 421 queries

📂 Loading pre-chunked corpus from notebook 2...
   Source: ../data/chunked_corpus
  ✅ Loaded 38909 pre-chunked chunks
  📋 Method used: semantic (2000/0.6)
   📈 Expansion: 2066 docs → 38909 chunks (18.83x)

🔢 Encoding chunks...


Batches:   0%|          | 0/2432 [00:00<?, ?it/s]

   ✅ Encoded 38909 chunks

🔍 Building FAISS index...
   ✅ Index built with 38909 vectors

🔤 Building BM25 index...
   ✅ BM25 index built
   ⚖️ Hybrid alpha: 0.55 (Dense) / 0.4 (BM25)

🎯 Processing queries...


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

   ✅ Encoded 421 queries
   📋 Retrieval config: top_k_retrieval=120, top_k_rerank=50

🔎 Retrieving and reranking...


...

   ✅ Encoded 21196 chunks

🔍 Building FAISS index...
   ✅ Index built with 21196 vectors

🔤 Building BM25 index...
   ✅ BM25 index built
   ⚖️ Hybrid alpha: 0.5 (Dense) / 0.5 (BM25)

🎯 Processing queries...


Batches:   0%|          | 0/104 [00:00<?, ?it/s]

   ✅ Encoded 1663 queries
   📋 Retrieval config: top_k_retrieval=150, top_k_rerank=60

🔎 Retrieving and reranking...


Retrieve+Rerank:   0%|          | 0/1663 [00:00<?, ?it/s]

   ✅ Generated 16630 results

📊 Evaluating...
   NDCG@10: 0.5048
   Queries evaluated: 1663

✅ Pipeline completed: 7/7 datasets


## 📊 Evaluation Summary

So sánh với baseline và đánh giá từng dataset (đặc biệt MultiHeirTT và TATQA):

In [12]:
if all_eval:
    print("\n" + "="*70)
    print("📊 EVALUATION SUMMARY (NDCG@10)")
    print("="*70)
    
    total_ndcg = 0
    total_queries = 0
    
    for ds, m in sorted(all_eval.items()):
        print(f"\n{ds.upper():15s}: {m['NDCG@10']:.4f} ({m['num_queries']} queries)")
        total_ndcg += m['NDCG@10'] * m['num_qrels']
        total_queries += m['num_qrels']
    
    if total_queries > 0:
        avg_ndcg = total_ndcg / total_queries
        print(f"\n{'='*70}")
        print(f"📈 WEIGHTED AVERAGE NDCG@10: {avg_ndcg:.4f}")
        print(f"{'='*70}")
        
        # Compare with baseline
        baseline = 0.328  # Original baseline
        gain = avg_ndcg - baseline
        gain_pct = (gain / baseline) * 100
        
        print(f"\n🎯 Performance vs Baseline:")
        print(f"   Baseline (no chunking): {baseline:.4f}")
        print(f"   Optimal chunking: {avg_ndcg:.4f}")
        print(f"   Improvement: +{gain:.4f} ({gain_pct:+.1f}%)")
        
        # Performance tier
        if avg_ndcg >= 0.58:
            print(f"\n🏆 EXCELLENT! Likely TOP 3 performance!")
        elif avg_ndcg >= 0.50:
            print(f"\n✅ VERY GOOD! Strong improvement!")
        elif avg_ndcg >= 0.40:
            print(f"\n✅ GOOD! Solid improvement!")
        else:
            print(f"\n⚠️ More tuning needed")
        
        print(f"\n{'='*70}")
else:
    print("\n⚠️ No evaluation metrics available")


📊 EVALUATION SUMMARY (NDCG@10)

CONVFINQA      : 0.4866 (421 queries)

FINANCEBENCH   : 0.7332 (150 queries)

FINDER         : 0.3903 (216 queries)

FINQA          : 0.4601 (1147 queries)

FINQABENCH     : 0.8662 (100 queries)

MULTIHEIRTT    : 0.1497 (974 queries)

TATQA          : 0.5048 (1663 queries)

📈 WEIGHTED AVERAGE NDCG@10: 0.4279

🎯 Performance vs Baseline:
   Baseline (no chunking): 0.3280
   Optimal chunking: 0.4279
   Improvement: +0.0999 (+30.5%)

✅ GOOD! Solid improvement!



## 💾 Generate Submission

In [13]:
if all_results:
    final_df = pd.concat(all_results, ignore_index=True)
    submission_df = final_df[['query_id', 'corpus_id']]
    
    submission_df.to_csv(CONFIG['output_file'], index=False)
    
    print(f"\n✅ Submission saved: {CONFIG['output_file']}")
    print(f"   Total entries: {len(submission_df):,}")
    print(f"   Unique queries: {submission_df['query_id'].nunique():,}")
    
    print(f"\n📋 Sample results:")
    print(submission_df.head(15))
    
    # Validation
    counts = submission_df.groupby('query_id').size()
    print(f"\n🔍 Validation:")
    print(f"   Results per query: {dict(counts.value_counts().sort_index())}")
    
    if (counts == 10).all():
        print(f"   ✅ All queries have exactly 10 results")
    else:
        print(f"   ⚠️ Warning: Some queries don't have 10 results")
        print(f"   Queries with != 10 results: {sum(counts != 10)}")
else:
    print("\n❌ No results to save")


✅ Submission saved: submission_optimal_chunking.csv
   Total entries: 46,710
   Unique queries: 4,671

📋 Sample results:
     query_id  corpus_id
0   qd4982518  dd4c4f7aa
1   qd4982518  dd4b88920
2   qd4982518  dd4b9f7f6
3   qd4982518  dd4bb5506
4   qd4982518  dd4b87d18
5   qd4982518  dd4be45d6
6   qd4982518  dd4bd3790
7   qd4982518  dd4b89cbc
8   qd4982518  dd4971510
9   qd4982518  dd4bd2250
10  qd49795a8  dd4befb5c
11  qd49795a8  dd4c05bc8
12  qd49795a8  dd4bd7b9c
13  qd49795a8  dd4979602
14  qd49795a8  dd4c4cf78

🔍 Validation:
   Results per query: {10: 4671}
   ✅ All queries have exactly 10 results


## 🎯 Final Summary

Tổng kết các cải tiến:
- **Semantic chunking** từ notebook 2 (thay thế fixed/recursive)
- **Dataset-specific optimizations** cho numerical reasoning datasets
- **Hybrid search** với tuned alpha per dataset

In [14]:
print("\n" + "="*70)
print("🎉 OPTIMAL CHUNKING PIPELINE COMPLETED!")
print("="*70)

print("\n✅ Key Features:")
print("   1. Semantic chunking (from notebook 2 evaluation)")
print("   2. Table-aware chunking")
print("   3. BGE-large embeddings")
print("   4. BGE-reranker-v2-m3 (SOTA)")
print("   5. Hybrid search (Dense + BM25)")
print("   6. Max-score aggregation")
print("   7. 🆕 Dataset-specific optimizations:")
print("      - MultiHeirTT: top_k=200, BM25-heavy (60%)")
print("      - TATQA: top_k=150, balanced hybrid (50-50)")
print("      - FinQA/ConvFinQA: top_k=120, slight BM25 boost")

if all_eval:
    total_ndcg = sum(m['NDCG@10'] * m['num_qrels'] for m in all_eval.values())
    total_queries = sum(m['num_qrels'] for m in all_eval.values())
    if total_queries > 0:
        avg = total_ndcg / total_queries
        print(f"\n📊 Final NDCG@10: {avg:.4f}")

print(f"\n💾 Output: {CONFIG['output_file']}")
print(f"\n🚀 Next: Upload to Kaggle!")
print("="*70)


🎉 OPTIMAL CHUNKING PIPELINE COMPLETED!

✅ Key Features:
   1. Semantic chunking (from notebook 2 evaluation)
   2. Table-aware chunking
   3. BGE-large embeddings
   4. BGE-reranker-v2-m3 (SOTA)
   5. Hybrid search (Dense + BM25)
   6. Max-score aggregation
   7. 🆕 Dataset-specific optimizations:
      - MultiHeirTT: top_k=200, BM25-heavy (60%)
      - TATQA: top_k=150, balanced hybrid (50-50)
      - FinQA/ConvFinQA: top_k=120, slight BM25 boost

📊 Final NDCG@10: 0.4279

💾 Output: submission_optimal_chunking.csv

🚀 Next: Upload to Kaggle!
